# Essentials of Statistics for Astronomers with Python

# **Statistical Methods for Astronomical Data Analysis**

This tutorial covers essential statistical concepts for astronomers:
1. Basic probability and distributions in astronomy
2. Error propagation and uncertainty analysis
3. Maximum Likelihood Estimation in astronomy
4. Hypothesis testing (e.g., detecting exoplanets)
5. Bayesian inference for astronomical problems
6. Monte Carlo simulations for error estimation
7. Correlation and regression with astronomical data
8. Survival analysis for censored data (non-detections)
9. Non-parametric statistics for astronomy
10. Multi-wavelength data analysis

## 1. Setup and Astronomical Context

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize, curve_fit
from scipy.stats import norm, poisson, chi2, uniform
import emcee
import corner
from astropy.stats import sigma_clip, biweight_location, biweight_scale
from astropy.modeling import models, fitting
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Astronomical constants
SOLAR_MASS = 1.989e30  # kg
SOLAR_RADIUS = 6.957e8  # m
PARSEC = 3.086e16  # m
H0 = 70  # km/s/Mpc (Hubble constant)

print("Statistical Astronomy Toolkit Ready!")
print(f"Using emcee version: {emcee.__version__}")

## 2. Probability Distributions in Astronomy

In [ ]:
# Generate astronomical datasets with different distributions
np.random.seed(42)

# 1. Normal distribution - measurement errors
measurement_errors = np.random.normal(0, 0.1, 1000)
# 2. Poisson distribution - photon counting
photon_counts = np.random.poisson(lam=50, size=1000)
# 3. Power-law distribution - galaxy luminosities
def power_law(x, alpha, xmin):
    return (alpha-1)/xmin * (x/xmin)**(-alpha)
galaxy_luminosities = np.random.power(2.5, 1000) * 1e10

# Plot distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Normal (Gaussian)
x = np.linspace(-0.5, 0.5, 100)
axes[0,0].hist(measurement_errors, bins=30, density=True, alpha=0.7, color='blue')
axes[0,0].plot(x, norm.pdf(x, 0, 0.1), 'r-', linewidth=2)
axes[0,0].set_xlabel('Measurement Error (mag)')
axes[0,0].set_ylabel('Probability Density')
axes[0,0].set_title('Gaussian Distribution - Photometric Errors')
axes[0,0].grid(True, alpha=0.3)

# Poisson
x = np.arange(20, 80)
axes[0,1].hist(photon_counts, bins=30, density=True, alpha=0.7, color='green')
axes[0,1].plot(x, poisson.pmf(x, 50), 'r-', linewidth=2, marker='o')
axes[0,1].set_xlabel('Photon Counts')
axes[0,1].set_ylabel('Probability')
axes[0,1].set_title('Poisson Distribution - Photon Statistics')
axes[0,1].grid(True, alpha=0.3)

# Power-law
log_bins = np.logspace(8, 11, 50)
axes[1,0].hist(galaxy_luminosities, bins=log_bins, density=True, alpha=0.7, color='orange')
axes[1,0].set_xscale('log')
axes[1,0].set_yscale('log')
axes[1,0].set_xlabel('Galaxy Luminosity (L☉)')
axes[1,0].set_ylabel('Probability Density')
axes[1,0].set_title('Power-law Distribution - Galaxy Luminosity Function')
axes[1,0].grid(True, alpha=0.3)

# Compare with normal distribution
axes[1,1].hist(galaxy_luminosities, bins=50, density=True, alpha=0.5, color='orange', label='Data')
axes[1,1].hist(np.random.normal(np.mean(galaxy_luminosities), np.std(galaxy_luminosities), 1000), 
               bins=50, density=True, alpha=0.5, color='blue', label='Normal fit')
axes[1,1].set_xlabel('Galaxy Luminosity (L☉)')
axes[1,1].set_ylabel('Probability Density')
axes[1,1].set_title('Power-law vs Normal Distribution')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Error Propagation in Astronomical Measurements

In [ ]:
# Example 1: Distance modulus error propagation
def distance_modulus(parallax_mas, error_parallax):
    """Calculate distance modulus from parallax with error propagation"""
    distance_pc = 1000 / parallax_mas
    error_distance = (1000 / parallax_mas**2) * error_parallax
    
    # Distance modulus: μ = 5 log10(d) - 5
    mu = 5 * np.log10(distance_pc) - 5
    error_mu = (5 / (distance_pc * np.log(10))) * error_distance
    
    return mu, error_mu

# Example measurements
parallax_values = np.array([10, 5, 2, 1])  # mas
parallax_errors = np.array([0.5, 0.3, 0.1, 0.05])  # mas

print("Distance Modulus with Errors:")
print("Parallax (mas) | Distance (pc) | Distance Modulus | Error")
print("-" * 60)
for p, e in zip(parallax_values, parallax_errors):
    mu, error_mu = distance_modulus(p, e)
    d = 1000 / p
    print(f"{p:12.1f} | {d:12.1f} | {mu:15.2f} | {error_mu:.2f}")

### Monte Carlo Error Propagation

In [ ]:
def monte_carlo_error_propagation(func, params, errors, n_simulations=10000):
    """Generic Monte Carlo error propagation"""
    simulated_results = []
    for _ in range(n_simulations):
        simulated_params = [np.random.normal(p, e) for p, e in zip(params, errors)]
        simulated_results.append(func(*simulated_params))
    return np.mean(simulated_results), np.std(simulated_results)

# Example: Magnitude to flux conversion with errors
def mag_to_flux(mag, mag_error):
    """Convert magnitude to flux (F = 10^(-0.4*mag))"""
    flux = 10**(-0.4 * mag)
    return flux

# Observed star with magnitude error
mag_obs = 12.5
mag_error = 0.05

# Analytical error propagation
flux = 10**(-0.4 * mag_obs)
analytical_error = np.abs(flux * (-0.4 * np.log(10))) * mag_error

# Monte Carlo error propagation
monte_carlo_flux, monte_carlo_error = monte_carlo_error_propagation(
    mag_to_flux, [mag_obs], [mag_error]
)

print(f"Flux: {flux:.4e}")
print(f"Analytical error: {analytical_error:.4e}")
print(f"Monte Carlo flux: {monte_carlo_flux:.4e} +/- {monte_carlo_error:.4e}")

## 4. Maximum Likelihood Estimation for Astronomical Data

In [ ]:
# Example: Fitting a power-law to galaxy counts
def power_law_likelihood(params, x, y, y_err):
    """Negative log-likelihood for power law: y = A * x^α"""
    A, alpha = params
    model = A * x**alpha
    likelihood = -0.5 * np.sum(((y - model) / y_err)**2 + np.log(2*np.pi*y_err**2))
    return -likelihood

# Generate simulated galaxy counts
np.random.seed(42)
x_galaxies = np.logspace(0, 2, 50)  # galaxy luminosity bins
true_A, true_alpha = 100, -1.5
y_true = true_A * x_galaxies**true_alpha
y_obs = y_true * np.random.normal(1, 0.1, len(x_galaxies))

# MLE fitting
initial_params = [50, -1]
result = minimize(power_law_likelihood, initial_params, args=(x_galaxies, y_obs, y_obs*0.1))
A_mle, alpha_mle = result.x

print("Power-law MLE Results:")
print(f"True A: {true_A:.1f}, MLE A: {A_mle:.1f}")
print(f"True α: {true_alpha:.2f}, MLE α: {alpha_mle:.2f}")

# Plot results
plt.figure(figsize=(10, 6))
plt.loglog(x_galaxies, y_true, 'k-', linewidth=2, label='True relation')
plt.errorbar(x_galaxies, y_obs, yerr=y_obs*0.1, fmt='o', alpha=0.5, label='Observed data')
plt.loglog(x_galaxies, A_mle * x_galaxies**alpha_mle, 'r--', linewidth=2, label='MLE fit')
plt.xlabel('Galaxy Luminosity (L☉)')
plt.ylabel('Number of Galaxies')
plt.title('Maximum Likelihood Estimation of Galaxy Luminosity Function')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Hypothesis Testing - Exoplanet Detection

In [ ]:
# Simulate radial velocity data with and without planet
np.random.seed(42)
time = np.linspace(0, 100, 50)
period = 12.5  # days
amplitude = 10  # m/s
noise_std = 2  # m/s

# Model with planet
rv_signal = amplitude * np.sin(2 * np.pi * time / period)
rv_obs_planet = rv_signal + np.random.normal(0, noise_std, len(time))

# Model without planet (only noise)
rv_obs_noplanet = np.random.normal(0, noise_std, len(time))

# Hypothesis testing: H0 = no planet, H1 = planet exists
def rv_model(t, amplitude, period):
    return amplitude * np.sin(2 * np.pi * t / period)

# Fit planet model to data
popt, pcov = curve_fit(rv_model, time, rv_obs_planet, p0=[1, 10])
rv_fit = rv_model(time, *popt)

# Calculate chi-square for both hypotheses
chi2_noplanet = np.sum(((rv_obs_planet - 0) / noise_std)**2)
chi2_planet = np.sum(((rv_obs_planet - rv_fit) / noise_std)**2)
chi2_diff = chi2_noplanet - chi2_planet

# F-test
df_diff = 2  # Two additional parameters (amplitude, period)
f_stat = (chi2_diff / df_diff) / (chi2_planet / (len(time) - 3))
p_value = 1 - stats.f.cdf(f_stat, df_diff, len(time) - 3)

print("Exoplanet Detection Hypothesis Test:")
print(f"Chi² (no planet): {chi2_noplanet:.1f}")
print(f"Chi² (planet): {chi2_planet:.1f}")
print(f"Chi² improvement: {chi2_diff:.1f}")
print(f"F-statistic: {f_stat:.2f}")
print(f"p-value: {p_value:.4e}")
print(f"Planet detection significance: {p_value < 0.001} (p < 0.001)")

# Visualize
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.errorbar(time, rv_obs_noplanet, yerr=noise_std, fmt='o', alpha=0.5)
plt.xlabel('Time (days)')
plt.ylabel('Radial Velocity (m/s)')
plt.title('No Planet Hypothesis - Pure Noise')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.errorbar(time, rv_obs_planet, yerr=noise_std, fmt='o', alpha=0.5, label='Data')
plt.plot(time, rv_fit, 'r-', linewidth=2, label='Planet model')
plt.xlabel('Time (days)')
plt.ylabel('Radial Velocity (m/s)')
plt.title(f'Planet Hypothesis - P = {popt[1]:.1f} days')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Bayesian Inference for Astronomical Problems

In [ ]:
# Bayesian inference for supernova cosmology
def supernova_likelihood(params, z, mu_obs, mu_err):
    """Likelihood for cosmological parameters"""
    H0, Omega_M = params
    # Simplified distance modulus for flat universe
    def distance_modulus(z, H0, Omega_M):
        # Comoving distance integral (simplified)
        integral = np.cumsum(1/np.sqrt(Omega_M*(1+z_vals)**3 + (1-Omega_M))) * np.diff(np.concatenate([[0], z]))
        dL = (1+z) * integral * (299792 / H0)
        mu = 5 * np.log10(dL) - 5
        return mu
    
    z_vals = np.sort(z)
    mu_model = distance_modulus(z_vals, H0, Omega_M)
    
    # Interpolate for each observation
    mu_model_interp = np.interp(z, z_vals, mu_model)
    
    # Calculate likelihood
    chi2 = np.sum(((mu_obs - mu_model_interp) / mu_err)**2)
    return -0.5 * chi2

# Simulate supernova data
np.random.seed(42)
z_sn = np.random.uniform(0.01, 0.5, 30)
true_H0, true_Omega_M = 70, 0.3
mu_true = 5 * np.log10((1+z_sn) * (299792/true_H0) * 
                       (2/(true_Omega_M**2) * (true_Omega_M*z_sn + (true_Omega_M-1)*(np.sqrt(1+true_Omega_M*z_sn)-1)))) - 5
mu_obs = mu_true + np.random.normal(0, 0.2, len(z_sn))

# MCMC sampling using emcee
def log_probability(params, z, mu_obs, mu_err):
    H0, Omega_M = params
    if H0 < 50 or H0 > 90 or Omega_M < 0 or Omega_M > 1:
        return -np.inf
    return supernova_likelihood(params, z, mu_obs, mu_err)

# Initialize MCMC
n_walkers = 32
n_dim = 2
initial_pos = [np.array([70, 0.3]) + 1e-4 * np.random.randn(n_dim) for _ in range(n_walkers)]
sampler = emcee.EnsembleSampler(n_walkers, n_dim, log_probability, args=(z_sn, mu_obs, 0.2))

# Run MCMC
print("Running MCMC for cosmological parameter estimation...")
sampler.run_mcmc(initial_pos, 2000, progress=True)
samples = sampler.get_chain(discard=500, flat=True)

# Plot results
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Trace plots
axes[0,0].plot(sampler.get_chain()[:,:,0].T, alpha=0.3)
axes[0,0].set_ylabel('H₀ (km/s/Mpc)')
axes[0,0].set_title('MCMC Chain - Hubble Constant')

axes[0,1].plot(sampler.get_chain()[:,:,1].T, alpha=0.3)
axes[0,1].set_ylabel('Ω_M')
axes[0,1].set_title('MCMC Chain - Matter Density')

# Corner plot
corner.corner(samples, labels=['H₀ (km/s/Mpc)', 'Ω_M'], truths=[true_H0, true_Omega_M],
              quantiles=[0.16, 0.5, 0.84], show_titles=True, title_fmt='.2f',
              fig=fig)

plt.tight_layout()
plt.show()

# Results
h0_median = np.median(samples[:,0])
h0_std = np.std(samples[:,0])
omega_median = np.median(samples[:,1])
omega_std = np.std(samples[:,1])

print("Bayesian Inference Results:")
print(f"H₀ = {h0_median:.1f} ± {h0_std:.1f} km/s/Mpc")
print(f"Ω_M = {omega_median:.3f} ± {omega_std:.3f}")

## 7. Monte Carlo Simulations for Error Estimation

In [ ]:
# Monte Carlo simulation for galaxy cluster mass estimation
def estimate_cluster_mass(velocity_dispersion, r_virial, n_galaxies=100):
    """Estimate cluster mass from velocity dispersion"""
    G = 4.300e-6  # kpc (km/s)^2 / M☉
    mass = (velocity_dispersion**2 * r_virial) / G
    # Add measurement uncertainty
    mass_err = mass / np.sqrt(n_galaxies)
    return mass, mass_err

# Generate cluster observations
np.random.seed(42)
n_realizations = 10000
true_mass = 1e15  # M☉

# Simulate 100 galaxy velocity measurements
v_disp_obs = np.random.normal(800, 100, n_realizations)  # km/s
r_vir_obs = np.random.normal(2000, 200, n_realizations)  # kpc

# Calculate masses with uncertainties
masses = []
mass_errors = []
for vd, rv in zip(v_disp_obs, r_vir_obs):
    m, m_err = estimate_cluster_mass(vd, rv)
    masses.append(m)
    mass_errors.append(m_err)

masses = np.array(masses)
mass_errors = np.array(mass_errors)

# Plot mass distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(masses, bins=50, alpha=0.7, color='purple', edgecolor='black')
plt.axvline(true_mass, color='red', linestyle='--', linewidth=2, label=f'True mass = {true_mass:.0e} M☉')
plt.axvline(np.median(masses), color='blue', linestyle='-', linewidth=2, label=f'Median = {np.median(masses):.0e}')
plt.xlabel('Cluster Mass (M☉)')
plt.ylabel('Frequency')
plt.title('Monte Carlo Mass Distribution')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(mass_errors, bins=50, alpha=0.7, color='orange', edgecolor='black')
plt.axvline(np.median(mass_errors), color='red', linestyle='--', 
           label=f'Median error = {np.median(mass_errors):.1e}')
plt.xlabel('Mass Error (M☉)')
plt.ylabel('Frequency')
plt.title('Mass Uncertainty Distribution')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Monte Carlo Mass Estimates:")
print(f"Input true mass: {true_mass:.1e} M☉")
print(f"Mean estimated mass: {np.mean(masses):.1e} M☉")
print(f"Median estimated mass: {np.median(masses):.1e} M☉")
print(f"68% confidence interval: [{np.percentile(masses, 16):.1e}, {np.percentile(masses, 84):.1e}]")

## 8. Correlation and Regression in Astronomy

In [ ]:
# Tully-Fisher relation for spiral galaxies
np.random.seed(42)
n_galaxies_tf = 100

# True Tully-Fisher relation: M = a * log10(V) + b
true_a, true_b = -7.5, -2.0
v_rot = np.random.uniform(100, 300, n_galaxies_tf)  # km/s
log_v = np.log10(v_rot)
true_magnitude = true_a * log_v + true_b

# Add observational scatter
observed_magnitude = true_magnitude + np.random.normal(0, 0.5, n_galaxies_tf)

# Perform regression with measurement errors
def tf_regression(log_v, v_err, mag, mag_err):
    """Fit Tully-Fisher relation accounting for errors in both axes"""
    # Simple linear regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(log_v, mag)
    return slope, intercept, r_value**2

slope, intercept, r2 = tf_regression(log_v, 0.05, observed_magnitude, 0.2)

# Fit with robust statistics
from scipy import odr

def linear_func(p, x):
    return p[0]*x + p[1]

# Orthogonal distance regression (accounts for errors in both axes)
data = odr.RealData(log_v, observed_magnitude, sx=0.05, sy=0.2)
odr_model = odr.Model(linear_func)
odr_fit = odr.ODR(data, odr_model, beta0=[-7, -2])
odr_output = odr_fit.run()
odr_slope, odr_intercept = odr_output.beta

print("Tully-Fisher Relation Regression Results:")
print(f"True relation: M = {true_a:.2f} log10(V) + {true_b:.2f}")
print(f"OLS: M = {slope:.2f} log10(V) + {intercept:.2f}, R² = {r2:.3f}")
print(f"ODR: M = {odr_slope:.2f} log10(V) + {odr_intercept:.2f}")

# Plot
plt.figure(figsize=(10, 8))
plt.errorbar(log_v, observed_magnitude, xerr=0.05, yerr=0.2, fmt='o', alpha=0.5, label='Observations')
v_fit = np.linspace(2, 2.5, 100)
plt.plot(v_fit, slope*v_fit + intercept, 'r-', linewidth=2, label=f'OLS fit (R²={r2:.3f})')
plt.plot(v_fit, odr_slope*v_fit + odr_intercept, 'g--', linewidth=2, label='ODR fit')
plt.plot(v_fit, true_a*v_fit + true_b, 'k:', linewidth=2, label='True relation')
plt.xlabel('log10(Rotation Velocity [km/s])')
plt.ylabel('Absolute Magnitude')
plt.title('Tully-Fisher Relation for Spiral Galaxies')
plt.gca().invert_yaxis()
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 9. Survival Analysis for Astronomical Non-Detections

In [ ]:
# Simulate galaxy survey with flux upper limits
np.random.seed(42)
n_galaxies_survey = 200

# True flux distribution (power-law)
true_flux = np.random.power(2, n_galaxies_survey) * 100
detection_limit = 10  # mJy

# Create observed dataset with censoring
observed_flux = true_flux.copy()
is_detected = true_flux >= detection_limit
observed_flux[~is_detected] = detection_limit / 2  # Upper limit placeholder

# Kaplan-Meier estimator for survival function
from lifelines import KaplanMeierFitter

# Create survival dataset
kmf = KaplanMeierFitter()
# Event: detected (1) or not (0)
event_observed = is_detected.astype(int)

# Fit Kaplan-Meier
kmf.fit(observed_flux, event_observed=event_observed, label='Flux distribution')

# Plot survival function
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Flux distribution
axes[0].hist(true_flux, bins=30, alpha=0.5, color='blue', label='True flux (all)')
axes[0].hist(observed_flux[is_detected], bins=30, alpha=0.5, color='green', label='Detected')
axes[0].axvline(detection_limit, color='red', linestyle='--', linewidth=2, label='Detection limit')
axes[0].set_xlabel('Flux (mJy)')
axes[0].set_ylabel('Number of Galaxies')
axes[0].set_title('Galaxy Flux Distribution with Detection Limit')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Kaplan-Meier survival curve
kmf.plot_survival_function(ax=axes[1])
axes[1].set_xlabel('Flux (mJy)')
axes[1].set_ylabel('Survival Probability (P(flux > x))')
axes[1].set_title('Kaplan-Meier Survival Function')
axes[1].axvline(detection_limit, color='red', linestyle='--', linewidth=2, label='Detection limit')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Survey Statistics:")
print(f"Total galaxies: {n_galaxies_survey}")
print(f"Detected galaxies: {is_detected.sum()}")
print(f"Detection rate: {is_detected.sum()/n_galaxies_survey*100:.1f}%")
print(f"Median flux (Kaplan-Meier): {kmf.median_survival_time_:.2f} mJy")

## 10. Non-Parametric Statistics for Astronomy

In [ ]:
# Compare two star-forming region samples
np.random.seed(42)

# Sample 1: Young stellar clusters
young_clusters = np.random.normal(1, 0.5, 50)  # Myr
# Sample 2: Old stellar clusters
old_clusters = np.random.normal(10, 3, 50)  # Myr

# Add some outliers (merging clusters)
young_clusters = np.append(young_clusters, [5, 6, 7])
old_clusters = np.append(old_clusters, [2, 3])

# Kolmogorov-Smirnov test
ks_statistic, ks_pvalue = stats.ks_2samp(young_clusters, old_clusters)

# Mann-Whitney U test
u_statistic, u_pvalue = stats.mannwhitneyu(young_clusters, old_clusters)

print("Non-parametric Tests for Cluster Ages:")
print(f"KS test: statistic = {ks_statistic:.3f}, p-value = {ks_pvalue:.4f}")
print(f"Mann-Whitney U test: statistic = {u_statistic:.1f}, p-value = {u_pvalue:.4f}")
print(f"Significant difference: {ks_pvalue < 0.05}")

# Plot distributions
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(young_clusters, bins=15, alpha=0.5, color='blue', label='Young clusters', density=True)
plt.hist(old_clusters, bins=15, alpha=0.5, color='red', label='Old clusters', density=True)
plt.xlabel('Age (Myr)')
plt.ylabel('Density')
plt.title('Age Distribution of Star Clusters')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot([young_clusters, old_clusters], labels=['Young', 'Old'])
plt.ylabel('Age (Myr)')
plt.title('Box Plot Comparison')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Robust Statistics for Outlier-Prone Data

In [ ]:
# Simulate galaxy redshift survey with outliers
np.random.seed(42)

# Main galaxy population (z ~ 0.1)
n_galaxies_robust = 200
z_main = np.random.normal(0.1, 0.02, n_galaxies_robust)

# Contaminating outliers (foreground/background galaxies)
n_outliers = 20
z_outliers = np.random.uniform(0, 0.5, n_outliers)

z_all = np.concatenate([z_main, z_outliers])
print(f"Total galaxies: {len(z_all)}")
print(f"Outliers: {n_outliers}")

# Compare different estimators
mean_z = np.mean(z_all)
median_z = np.median(z_all)
biweight_z = biweight_location(z_all)
std_z = np.std(z_all)
mad_z = stats.median_abs_deviation(z_all)
biweight_scale_z = biweight_scale(z_all)

print("\nCentral Tendency Estimators:")
print(f"Mean: {mean_z:.4f}")
print(f"Median: {median_z:.4f}")
print(f"Biweight Location: {biweight_z:.4f}")
print(f"True value (without outliers): {np.mean(z_main):.4f}")

print("\nScale Estimators:")
print(f"Standard deviation: {std_z:.4f}")
print(f"Median Absolute Deviation: {mad_z:.4f}")
print(f"Biweight Scale: {biweight_scale_z:.4f}")

# Visualize
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(z_main, bins=30, alpha=0.5, color='blue', label='Main population')
plt.hist(z_outliers, bins=20, alpha=0.5, color='red', label='Outliers')
plt.axvline(mean_z, color='green', linestyle='-', linewidth=2, label=f'Mean: {mean_z:.3f}')
plt.axvline(median_z, color='orange', linestyle='--', linewidth=2, label=f'Median: {median_z:.3f}')
plt.axvline(biweight_z, color='purple', linestyle=':', linewidth=2, label=f'Biweight: {biweight_z:.3f}')
plt.xlabel('Redshift')
plt.ylabel('Frequency')
plt.title('Galaxy Redshift Distribution with Outliers')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# Show impact of outliers on mean
n_outliers_range = np.arange(0, 50)
means = []
for n in n_outliers_range:
    outliers = np.random.uniform(0, 0.5, n)
    data = np.concatenate([z_main, outliers])
    means.append(np.mean(data))
    
plt.plot(n_outliers_range, means, 'b-', linewidth=2)
plt.axhline(np.mean(z_main), color='r', linestyle='--', label='True mean (no outliers)')
plt.xlabel('Number of Outliers')
plt.ylabel('Sample Mean')
plt.title('Sensitivity of Mean to Outliers')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Multi-Wavelength Data Analysis

In [ ]:
# Simulate SED (Spectral Energy Distribution) of a galaxy
def blackbody(wavelength, temperature, normalization):
    """Blackbody flux density"""
    h = 6.626e-27  # erg s
    c = 3e10  # cm/s
    k = 1.38e-16  # erg/K
    wavelength_cm = wavelength * 1e-4  # Convert to cm
    B = (2*h*c**2/wavelength_cm**5) / (np.exp(h*c/(wavelength_cm*k*temperature)) - 1)
    return B * normalization

def power_law_sed(wavelength, alpha, norm):
    """Power-law SED for radio/X-ray"""
    return norm * wavelength**alpha

# Generate multi-wavelength data
np.random.seed(42)
wavelengths = np.logspace(0, 5, 50)  # Angstroms (1e0 to 1e5)

# UV/Optical (stellar component)
optical_flux = blackbody(wavelengths, 5000, 1e-10)
# IR (dust component)
ir_flux = blackbody(wavelengths, 200, 1e-9)
# Radio (synchrotron)
radio_flux = power_law_sed(wavelengths, -0.7, 1e-12)

total_flux = optical_flux + ir_flux + radio_flux
# Add noise
observed_flux = total_flux * np.random.normal(1, 0.1, len(wavelengths))

# Convert to magnitudes
observed_mag = -2.5 * np.log10(observed_flux)

# Fit SED components
from scipy.optimize import curve_fit

def galaxy_sed(wavelength, T_star, T_dust, radio_norm):
    """Simple 3-component galaxy SED model"""
    optical = blackbody(wavelength, T_star, 1e-10)
    infrared = blackbody(wavelength, T_dust, 1e-9)
    radio = radio_norm * wavelength**(-0.7)
    return optical + infrared + radio

# Fit model
popt, pcov = curve_fit(galaxy_sed, wavelengths, observed_flux, p0=[5000, 200, 1e-12])
T_star_fit, T_dust_fit, radio_norm_fit = popt

print("Multi-wavelength SED Analysis:")
print(f"Stellar temperature: {T_star_fit:.0f} K (input: 5000 K)")
print(f"Dust temperature: {T_dust_fit:.0f} K (input: 200 K)")
print(f"Radio normalization: {radio_norm_fit:.2e} (input: 1e-12)")

# Plot SED
plt.figure(figsize=(12, 8))
plt.loglog(wavelengths, observed_flux, 'bo', alpha=0.5, label='Observed')
plt.loglog(wavelengths, optical_flux, 'g--', linewidth=2, alpha=0.7, label='Stellar (optical)')
plt.loglog(wavelengths, ir_flux, 'r--', linewidth=2, alpha=0.7, label='Dust (IR)')
plt.loglog(wavelengths, radio_flux, 'm--', linewidth=2, alpha=0.7, label='Synchrotron (radio)')
plt.loglog(wavelengths, galaxy_sed(wavelengths, *popt), 'k-', linewidth=2, label='Best fit')
plt.xlabel('Wavelength (Angstroms)')
plt.ylabel('Flux Density (erg/s/cm²/Å)')
plt.title('Multi-wavelength SED of a Galaxy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 13. Goodness-of-Fit Tests for Astronomical Models

In [ ]:
# Test if observed periodogram matches expected noise distribution
def periodogram(frequencies, amplitudes):
    """Simple Lomb-Scargle like periodogram"""
    return amplitudes**2 / len(frequencies)

# Simulate time series of variable star
np.random.seed(42)
time_series = np.linspace(0, 1000, 500)
true_period = 50  # days
signal = 0.5 * np.sin(2 * np.pi * time_series / true_period)
noise = np.random.normal(0, 0.2, len(time_series))
observed = signal + noise

# Simple periodogram
frequencies = np.linspace(0.01, 0.5, 200)
periodogram_power = []

for f in frequencies:
    model = np.sin(2 * np.pi * f * time_series)
    power = np.corrcoef(observed, model)[0,1]**2
    periodogram_power.append(power)

periodogram_power = np.array(periodogram_power)

# Goodness-of-fit: Compare with noise-only distribution
# Generate noise-only periodograms
n_simulations = 1000
noise_periodograms = []

for _ in range(n_simulations):
    noise_sim = np.random.normal(0, 0.2, len(time_series))
    noise_power = []
    for f in frequencies:
        model = np.sin(2 * np.pi * f * time_series)
        power = np.corrcoef(noise_sim, model)[0,1]**2
        noise_power.append(power)
    noise_periodograms.append(np.max(noise_power))

noise_periodograms = np.array(noise_periodograms)
observed_max_power = np.max(periodogram_power)

# False alarm probability
fap = np.sum(noise_periodograms >= observed_max_power) / n_simulations

print("Periodogram Significance Test:")
print(f"Observed maximum power: {observed_max_power:.3f}")
print(f"False Alarm Probability: {fap:.3e}")
print(f"Significant detection: {fap < 0.01}")

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(frequencies, periodogram_power, 'b-', linewidth=1)
plt.axhline(observed_max_power, color='r', linestyle='--', linewidth=2, label=f'Peak: {observed_max_power:.3f}')
plt.xlabel('Frequency (1/day)')
plt.ylabel('Power')
plt.title('Lomb-Scargle Periodogram')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(noise_periodograms, bins=50, alpha=0.7, color='green', edgecolor='black', density=True)
plt.axvline(observed_max_power, color='r', linestyle='--', linewidth=2, label=f'Observed (FAP={fap:.0e})')
plt.xlabel('Maximum Power')
plt.ylabel('Probability Density')
plt.title('Noise Distribution of Maximum Power')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 14. Practical Case Study: RR Lyrae Star Classification

In [ ]:
# Generate RR Lyrae light curve data
np.random.seed(42)
n_observations = 200

# RR Lyrae characteristics
period_rr = 0.5  # days
amplitude_rr = 0.8  # magnitudes
mean_mag = 14.0
phase = np.random.uniform(0, period_rr, n_observations)

# Light curve model (sawtooth for RR Lyrae)
def rr_lyrae_light_curve(phase, period, amplitude, mean_mag):
    p_norm = phase / period
    # Asymmetric sawtooth typical of RR Lyrae
    mag = mean_mag + amplitude * (2 * p_norm - 1)**3
    return mag

# Generate with noise
true_light_curve = rr_lyrae_light_curve(phase, period_rr, amplitude_rr, mean_mag)
observed_light_curve = true_light_curve + np.random.normal(0, 0.05, n_observations)

# Statistical analysis
# Check for periodic signal
phases_for_fit = np.linspace(0, period_rr, 100)
template_lc = rr_lyrae_light_curve(phases_for_fit, period_rr, amplitude_rr, mean_mag)

# Cross-correlation
from scipy.signal import correlate
correlation = correlate(template_lc, observed_light_curve)

# Statistical tests
# Shapiro-Wilk test for normality of residuals
residuals = observed_light_curve - true_light_curve
shapiro_stat, shapiro_p = stats.shapiro(residuals)

# Check if folded light curve is non-constant (ANOVA)
folded_phases = phase / period_rr
f_stat, anova_p = stats.f_oneway(*[observed_light_curve[folded_phases > i/5] for i in range(5)])

print("RR Lyrae Statistical Analysis:")
print(f"Number of observations: {n_observations}")
print(f"Phase coverage: {len(np.unique(folded_phases))} unique phases")
print(f"Residual normality test (Shapiro-Wilk): p-value = {shapiro_p:.3f}")
print(f"Phase-dependent variation (ANOVA): p-value = {anova_p:.2e}")
print(f"Star classification: {'RR Lyrae' if anova_p < 0.01 else 'Non-variable'}")

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.errorbar(phase, observed_light_curve, yerr=0.05, fmt='o', alpha=0.5, label='Observations')
plt.plot(phases_for_fit, template_lc, 'r-', linewidth=2, label='RR Lyrae template')
plt.xlabel('Phase (days)')
plt.ylabel('Magnitude')
plt.title('RR Lyrae Light Curve')
plt.gca().invert_yaxis()
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(residuals, bins=20, alpha=0.7, color='purple', edgecolor='black', density=True)
x = np.linspace(-0.2, 0.2, 100)
plt.plot(x, norm.pdf(x, 0, np.std(residuals)), 'r-', linewidth=2, label='Normal fit')
plt.xlabel('Residuals (mag)')
plt.ylabel('Density')
plt.title(f'Residual Distribution (p={shapiro_p:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 15. Summary and Best Practices

Print summary of statistical methods for astronomy

summary_text = """
Statistical Essentials for Astronomers - Summary

# Key Statistical Methods Covered:

## 1. Probability Distributions
- **Normal**: Measurement errors, noise
- **Poisson**: Photon counting, rare events
- **Power-law**: Galaxy luminosity function, mass distributions

## 2. Error Analysis
- Error propagation formulas for derived quantities
- Monte Carlo simulations for complex systems
- Robust statistics for outlier handling

## 3. Hypothesis Testing
- F-test for model comparison (exoplanet detection)
- Chi-square goodness-of-fit
- False alarm probability for periodic signals

## 4. Bayesian Inference
- Parameter estimation with MCMC
- Prior selection and posterior distributions
- Model comparison with Bayes factors

## 5. Regression Methods
- Tully-Fisher and other scaling relations
- Orthogonal distance regression (errors in both axes)
- Robust regression techniques

## 6. Survival Analysis
- Handling upper limits and non-detections
- Kaplan-Meier estimator for flux distributions
- Censored data in luminosity functions

## 7. Non-parametric Statistics
- Kolmogorov-Smirnov test for distribution comparison
- Mann-Whitney U test for medians
- Kruskal-Wallis test for multiple groups

# Best Practices for Astronomical Statistics:

1. **Always propagate errors** - Use Monte Carlo when analytical methods are complex
2. **Check assumptions** - Test for normality, homoscedasticity before parametric tests
3. **Report uncertainties** - Include both statistical and systematic errors
4. **Use robust statistics** - Biweight estimators for outlier-prone data
5. **Consider detection limits** - Account for censoring in flux-limited surveys
6. **Visualize your data** - Always plot residuals and distributions
7. **Validate with simulations** - Test statistical methods on simulated data
8. **Report significance properly** - Include false alarm probabilities for detections

# Recommended Resources:
- "Statistics, Data Mining, and Machine Learning in Astronomy" (Ivezić et al.)
- "AstroML" - Python library for astronomical statistics
- "Bayesian Models for Astrophysical Data" (Hilbe et al.)
"""

print(summary_text)

In [ ]:
print("""
================================================================================
Tutorial Complete! You've learned essential statistical methods for astronomy.
Practice with real astronomical datasets to reinforce these concepts.
================================================================================
""")

## Appendix: Quick Reference - Statistical Functions in Python

In [ ]:
# Quick reference table as a DataFrame
stats_functions = pd.DataFrame({
    'Method': [
                'Mean', 'Median', 'Biweight location',
                'Std', 'MAD', 'Biweight scale',
                'Normal distribution', 'Poisson distribution',
                't-test', 'F-test', 'Chi-square test', 'KS test',
                'Linear regression', 'ODR regression',
                'MCMC (emcee)', 'Kaplan-Meier'
    ],
    'Function': [
                'np.mean()', 'np.median()', 'astroML.biweight_location()',
                'np.std()', 'stats.median_abs_deviation()', 'astroML.biweight_scale()',
                'scipy.stats.norm', 'scipy.stats.poisson',
                'scipy.stats.ttest_ind()', 'scipy.stats.f_oneway()', 'scipy.stats.chisquare()', 'scipy.stats.ks_2samp()',
                'scipy.stats.linregress()', 'scipy.odr.ODR()',
                'emcee.EnsembleSampler()', 'lifelines.KaplanMeierFitter()'
    ],
    'Use Case': [
                'Central value (normal data)', 'Central value (robust)', 'Central value (very robust)',
                'Spread (normal)', 'Spread (robust)', 'Spread (very robust)',
                'Measurement errors', 'Photon/event counts',
                'Compare two samples', 'Compare multiple samples', 'Goodness-of-fit', 'Distribution comparison',
                'Fitting y ~ x', 'Fitting with errors in x and y',
                'Bayesian parameter estimation', 'Censored data analysis'
    ]
})

print("Quick Reference - Statistical Functions for Astronomy:")
print(stats_functions.to_string(index=False))